In [ ]:
!pip install -q transformers datasets peft trl accelerate \
  bitsandbytes safetensors

In [ ]:
from google.colab import files
from datasets import load_dataset

# Selecione os 3 arquivos de uma vez na janela de upload
uploaded = files.upload()
print("Enviados:", list(uploaded.keys()))

arquivos = {
    "train": "/content/train.jsonl",
    "validation": "/content/validation.jsonl",
    "test": "/content/test.jsonl",
}

dataset = load_dataset("json", data_files=arquivos)

for nome, conjunto in dataset.items():
    print(f"{nome}: {len(conjunto)} exemplos")

print("\nPrimeiro exemplo de treinamento:")
print(dataset["train"][0])

Saving test.jsonl to test (4).jsonl
Saving train.jsonl to train (4).jsonl
Saving validation.jsonl to validation (4).jsonl
Enviados: ['test (4).jsonl', 'train (4).jsonl', 'validation (4).jsonl']
train: 157 exemplos
validation: 20 exemplos
test: 20 exemplos

Primeiro exemplo de treinamento:
{'messages': [{'role': 'user', 'content': 'Determine a complexidade temporal e espacial:\n\nvoid somar_matrizes(int A[N][N], int B[N][N], int C[N][N], int n) {\n    for (int i = 0; i < n; i++) {\n        for (int j = 0; j < n; j++) {\n            C[i][j] = A[i][j] + B[i][j];\n        }\n    }\n}'}, {'role': 'assistant', 'content': 'ANÁLISE DE COMPLEXIDADE\n\nComplexidade temporal:\n   O(n²)\n\n2. Complexidade espacial auxiliar:\n   O(1)\n\n3. Análise do algoritmo:\n   Dois laços aninhados percorrem todas as n x n posições das matrizes realizando adições.\n\n4. Justificativa matemática:\n   T(n) = n * n * c = c * n² = O(n²). As matrizes de saída já foram alocadas externamente.\n\n5. Observações:\n   O 

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

modelo_id = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(modelo_id)

modelo = AutoModelForCausalLM.from_pretrained(
    modelo_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Modelo carregado!")
print("Dispositivo:", modelo.device)
print("Parâmetros:", sum(p.numel() for p in modelo.parameters()))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Modelo carregado!
Dispositivo: cuda:0
Parâmetros: 596049920


In [ ]:
def preparar_exemplo(exemplo):
    texto = tokenizer.apply_chat_template(
        exemplo["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )

    return {"text": texto}


dataset_formatado = dataset.map(preparar_exemplo)

print(dataset_formatado["train"][0]["text"])

<|im_start|>user
Determine a complexidade temporal e espacial:

void somar_matrizes(int A[N][N], int B[N][N], int C[N][N], int n) {
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            C[i][j] = A[i][j] + B[i][j];
        }
    }
}<|im_end|>
<|im_start|>assistant
<think>

</think>

ANÁLISE DE COMPLEXIDADE

Complexidade temporal:
   O(n²)

2. Complexidade espacial auxiliar:
   O(1)

3. Análise do algoritmo:
   Dois laços aninhados percorrem todas as n x n posições das matrizes realizando adições.

4. Justificativa matemática:
   T(n) = n * n * c = c * n² = O(n²). As matrizes de saída já foram alocadas externamente.

5. Observações:
   O parâmetro de entrada n representa a dimensão lateral da matriz.<|im_end|>



In [ ]:
tamanhos = []

for exemplo in dataset_formatado["train"]:
    tokens = tokenizer(
        exemplo["text"],
        add_special_tokens=False
    )["input_ids"]

    tamanhos.append(len(tokens))

print("Menor exemplo:", min(tamanhos))
print("Maior exemplo:", max(tamanhos))
print("Média:", sum(tamanhos) / len(tamanhos))

Menor exemplo: 160
Maior exemplo: 345
Média: 217.61146496815286


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

modelo = get_peft_model(modelo, config_lora)

modelo.print_trainable_parameters()

trainable params: 5,046,272 || all params: 601,096,192 || trainable%: 0.8395


In [ ]:

import torch

tokenizer.padding_side = "right"

def collator_loRA(exemplos):
    tamanho_maximo = max(
        len(exemplo["input_ids"])
        for exemplo in exemplos
    )

    input_ids = []
    attention_mask = []
    labels = []

    for exemplo in exemplos:
        padding = tamanho_maximo - len(exemplo["input_ids"])

        input_ids.append(
            exemplo["input_ids"]
            + [tokenizer.pad_token_id] * padding
        )

        attention_mask.append(
            exemplo["attention_mask"]
            + [0] * padding
        )

        labels.append(
            exemplo["labels"]
            + [-100] * padding
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(
            attention_mask, dtype=torch.long
        ),
        "labels": torch.tensor(labels, dtype=torch.long)
    }


print("Collator preparado!")


Collator preparado!


In [ ]:
print("dataset:", "dataset" in globals())
print("tokenizer:", "tokenizer" in globals())
print("modelo:", "modelo" in globals())
print("dataset_formatado:", "dataset_formatado" in globals())

dataset: True
tokenizer: True
modelo: True
dataset_formatado: True


In [ ]:
exemplo = dataset["train"][0]

prompt_texto = tokenizer.apply_chat_template(
    exemplo["messages"][:-1],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

completo_texto = tokenizer.apply_chat_template(
    exemplo["messages"],
    tokenize=False,
    add_generation_prompt=False,
    enable_thinking=False
)

print("=== PROMPT ===")
print(repr(prompt_texto))

print("\n=== CONVERSA COMPLETA (INÍCIO) ===")
print(repr(completo_texto[:500]))

print("\nO texto completo começa com o prompt?",
      completo_texto.startswith(prompt_texto))

=== PROMPT ===
'<|im_start|>user\nDetermine a complexidade temporal e espacial:\n\nvoid somar_matrizes(int A[N][N], int B[N][N], int C[N][N], int n) {\n    for (int i = 0; i < n; i++) {\n        for (int j = 0; j < n; j++) {\n            C[i][j] = A[i][j] + B[i][j];\n        }\n    }\n}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

=== CONVERSA COMPLETA (INÍCIO) ===
'<|im_start|>user\nDetermine a complexidade temporal e espacial:\n\nvoid somar_matrizes(int A[N][N], int B[N][N], int C[N][N], int n) {\n    for (int i = 0; i < n; i++) {\n        for (int j = 0; j < n; j++) {\n            C[i][j] = A[i][j] + B[i][j];\n        }\n    }\n}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nANÁLISE DE COMPLEXIDADE\n\nComplexidade temporal:\n   O(n²)\n\n2. Complexidade espacial auxiliar:\n   O(1)\n\n3. Análise do algoritmo:\n   Dois laços aninhados percorrem todas as n x n posições d'

O texto completo começa com o prompt? True


In [ ]:

def tokenizar_exemplo(exemplo):
    mensagens = exemplo["messages"]

    prompt_texto = tokenizer.apply_chat_template(
        mensagens[:-1],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    completo_texto = tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )

    if not completo_texto.startswith(prompt_texto):
        raise ValueError(
            "O prompt não corresponde ao início do texto completo."
        )

    # Tokeniza a conversa completa
    tokens = tokenizer(
        completo_texto,
        add_special_tokens=False
    )["input_ids"]

    # Tokeniza o prompt separadamente para obter seu comprimento
    tokens_prompt = tokenizer(
        prompt_texto,
        add_special_tokens=False
    )["input_ids"]

    # Encontra o prefixo de tokens que realmente coincide
    prefixo_comum = 0

    for a, b in zip(tokens, tokens_prompt):
        if a != b:
            break
        prefixo_comum += 1

    if prefixo_comum == 0:
        raise ValueError("Não foi possível identificar o início do prompt.")

    # Máscara: não calcular loss sobre o prompt
    labels = (
        [-100] * prefixo_comum
        + tokens[prefixo_comum:]
    )

    if len(tokens) > 512:
        raise ValueError(
            f"Exemplo com {len(tokens)} tokens: ultrapassou 512."
        )

    return {
        "input_ids": tokens,
        "attention_mask": [1] * len(tokens),
        "labels": labels
    }


dataset_tokenizado = dataset.map(
    tokenizar_exemplo,
    remove_columns=dataset["train"].column_names
)

print("Tokenização concluída!")

for nome, conjunto in dataset_tokenizado.items():
    print(f"{nome}: {len(conjunto)} exemplos")


Map:   0%|          | 0/157 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenização concluída!
train: 157 exemplos
validation: 20 exemplos
test: 20 exemplos


In [ ]:
exemplo = dataset_tokenizado["train"][0]

ids = exemplo["input_ids"]
labels = exemplo["labels"]

inicio_resposta = next(
    (i for i, valor in enumerate(labels) if valor != -100),
    None
)

print("Total de tokens:", len(ids))
print("Tokens ignorados:", labels.count(-100))
print("Tokens usados na loss:", len(labels) - labels.count(-100))
print("Início da resposta:", inicio_resposta)

print("\nInício do trecho usado no treinamento:")
print(tokenizer.decode(ids[inicio_resposta:inicio_resposta + 40]))

Total de tokens: 243
Tokens ignorados: 105
Tokens usados na loss: 138
Início da resposta: 105

Início do trecho usado no treinamento:
ANÁLISE DE COMPLEXIDADE

Complexidade temporal:
   O(n²)

2. Complexidade espacial auxiliar:
   O(1)

3. Análise do algorit


In [14]:
print("GPU disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

modelo.print_trainable_parameters()

print("\nTamanhos dos conjuntos:")
for nome in ["train", "validation", "test"]:
    print(f"{nome}: {len(dataset_tokenizado[nome])}")

print("\nColunas do conjunto tokenizado:")
print(dataset_tokenizado["train"].column_names)

GPU disponível: True
GPU: Tesla T4
trainable params: 5,046,272 || all params: 601,096,192 || trainable%: 0.8395

Tamanhos dos conjuntos:
train: 157
validation: 20
test: 20

Colunas do conjunto tokenizado:
['input_ids', 'attention_mask', 'labels']


In [17]:
from transformers import Trainer, TrainingArguments

modelo.config.use_cache = False
modelo.config.pad_token_id = tokenizer.pad_token_id

argumentos = TrainingArguments(
    output_dir="/content/complexity-qwen3-lora",

    num_train_epochs=5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=5,
    lr_scheduler_type="cosine",

    optim="adamw_torch",
    max_grad_norm=1.0,

    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=10,
    report_to="none",
    seed=42
)

trainer = Trainer(
    model=modelo,
    args=argumentos,
    train_dataset=dataset_tokenizado["train"],
    eval_dataset=dataset_tokenizado["validation"],
    data_collator=collator_loRA
)

print("Trainer preparado!")

Trainer preparado!


In [18]:
resultado = trainer.train()

Epoch,Training Loss,Validation Loss
1,1.092885,0.880702
2,0.743657,0.735571
3,0.582247,0.698901
4,0.502277,0.691722
5,0.439749,0.689750


In [19]:
import os

print("Melhor checkpoint:")
print(trainer.state.best_model_checkpoint)

print("\nMenor validation loss:")
print(trainer.state.best_metric)

# Pasta final para salvar o modelo
pasta_saida = "/content/complexity-qwen3-lora-final"

# Salva o modelo LoRA
trainer.save_model(pasta_saida)

# Salva o tokenizer
tokenizer.save_pretrained(pasta_saida)

# Salva o estado do treinamento
trainer.save_state()

print("\nModelo salvo em:", pasta_saida)
print("Arquivos:", os.listdir(pasta_saida))

Melhor checkpoint:
/content/complexity-qwen3-lora/checkpoint-100

Menor validation loss:
0.6897500157356262

Modelo salvo em: /content/complexity-qwen3-lora-final
Arquivos: ['chat_template.jinja', 'README.md', 'adapter_config.json', 'training_args.bin', 'tokenizer.json', 'tokenizer_config.json', 'adapter_model.safetensors']


In [20]:
import torch

modelo = trainer.model
modelo.eval()

def gerar_analise(mensagens, max_new_tokens=256):
    # Usa somente as mensagens de entrada, sem a resposta esperada
    prompt = tokenizer.apply_chat_template(
        mensagens[:-1],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    entradas = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(modelo.device)

    with torch.inference_mode():
        saida = modelo.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Remove os tokens correspondentes ao prompt
    novos_tokens = saida[0][entradas["input_ids"].shape[1]:]

    return tokenizer.decode(
        novos_tokens,
        skip_special_tokens=True
    ).strip()


# Testa os três primeiros exemplos
for i in range(3):
    exemplo = dataset["test"][i]
    mensagens = exemplo["messages"]

    resposta_esperada = mensagens[-1]["content"]
    resposta_gerada = gerar_analise(mensagens)

    print("=" * 70)
    print(f"EXEMPLO {i + 1}")
    print("=" * 70)

    print("\nCÓDIGO ENVIADO:")
    print(mensagens[0]["content"])

    print("\nRESPOSTA ESPERADA:")
    print(resposta_esperada)

    print("\nRESPOSTA GERADA PELO MODELO:")
    print(resposta_gerada)

EXEMPLO 1

CÓDIGO ENVIADO:
Qual a complexidade deste algoritmo?

int pot_binaria(int base, int exp) {
    int res = 1;
    while (exp > 0) {
        if (exp % 2 == 1) res *= base;
        base *= base;
        exp /= 2;
    }
    return res;
}

RESPOSTA ESPERADA:
ANÁLISE DE COMPLEXIDADE

1. Complexidade temporal:

   O(log n)

2. Complexidade espacial auxiliar:
   O(1)

3. Análise do algoritmo:
   O expoente (representado por n) é dividido por 2 em cada iteração enquanto a base é elevada ao quadrado.

4. Justificativa matemática:
   T(n) = c * log2(n) = O(log n), onde n é o valor do expoente exp.

5. Observações:
   Exponenciação rápida em tempo logarítmico.

RESPOSTA GERADA PELO MODELO:
ANÁLISE DE COMPLEXIDADE

Complexidade temporal:
   O(log exp)

2. Complexidade espacial auxiliar:
   O(1)

3. Análise do algoritmo:
   O algoritmo realiza uma série de divisões por 2 até reduzir a exponência a zero, executando no máximo log2(exp) iterações.

4. Justificativa matemática:
   T(n) = c * l

In [21]:
for i in range(3, 8):
    exemplo = dataset["test"][i]
    mensagens = exemplo["messages"]

    resposta_esperada = mensagens[-1]["content"]
    resposta_gerada = gerar_analise(mensagens)

    print("=" * 70)
    print(f"EXEMPLO {i + 1}")
    print("=" * 70)

    print("\nCÓDIGO ENVIADO:")
    print(mensagens[0]["content"])

    print("\nRESPOSTA ESPERADA:")
    print(resposta_esperada)

    print("\nRESPOSTA GERADA:")
    print(resposta_gerada)

EXEMPLO 4

CÓDIGO ENVIADO:
Analise a complexidade espacial deste código:

int *vetor = malloc(n * sizeof(int));
if (vetor == NULL) return 1;
free(vetor);

RESPOSTA ESPERADA:
ANÁLISE DE COMPLEXIDADE

1. Complexidade temporal: O(1), considerando apenas as operações de alocação e liberação como operações primitivas.

2. Complexidade espacial auxiliar: O(n).

3. Análise do algoritmo: O programa solicita memória suficiente para armazenar n elementos inteiros. A quantidade de memória alocada cresce linearmente com n.

4. Justificativa matemática: O espaço alocado é n*sizeof(int) bytes. Como sizeof(int) é constante para uma implementação, o crescimento é proporcional a n.

5. Observações: O tempo real de malloc depende do alocador e não deve ser confundido com o custo de inicializar n posições.

RESPOSTA GERADA:
ANÁLISE DE COMPLEXIDADE

1. Complexidade temporal:

   O(1)

2. Complexidade espacial auxiliar:
   O(1)

3. Análise do algoritmo:
   Aloca um vetor dinâmico de tamanho n em memória e 